# LangChain (open source): Chains

### Outline
- `LLMChain` equivalent
- Sequential chains: `SimpleSequentialChain` and `SequentialChain` equivalents
- Router chain equivalent

Open-source recode of `02-LangChain-for-LLM-Application-Development/L3-chains.ipynb`.
`LLMChain`, `SimpleSequentialChain`, `SequentialChain`, `MultiPromptChain` and
`LLMRouterChain` were all removed - `langchain.chains` doesn't exist anymore in this LangChain
version. LCEL (the `|` operator) is now the only way to build chains:

- `LLMChain` -> `prompt | model | StrOutputParser()`.
- `SimpleSequentialChain` (single in, single out) -> pipe one chain's string output straight
  into the next chain's input dict.
- `SequentialChain` (multiple named inputs/outputs) -> a stack of
  `RunnablePassthrough.assign(key=chain)` calls, each adding one more key to a running dict.
- The router chain -> `model.with_structured_output(RouterDecision)` picks a destination name
  (LLM-as-classifier instead of a hand-parsed markdown JSON blob), then a plain Python dict
  lookup dispatches to the chosen chain.

`Data.csv` from the original isn't included in this repo - `data/reviews.csv` is a small
synthetic stand-in with the same shape (Product, Review columns, one review in French to
exercise the translate/detect-language chains).

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`open_source_agentic_ai_course`](../../open_source_agentic_ai_course/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [1]:
import sys
from pathlib import Path

# common.py / tracing.py live in open_source_agentic_ai_course/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../open_source_agentic_ai_course").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

import pandas as pd
from common import get_model, traced
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

In [2]:
df = pd.read_csv(COURSE_DIR / "data" / "reviews.csv")
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


In [4]:
model = get_model(temperature=0.9)

## `LLMChain` equivalent

`prompt | model | StrOutputParser()` - a prompt template piped into a model, piped into a
string-output parser.

In [11]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain = prompt | model | StrOutputParser()

product = "Queen Size Sheet Set"
my_response = chain.invoke({"product": product}, config=traced("L3: LLMChain equivalent"))
print(my_response)

Below are seven brand‑name ideas that instantly tell your audience *what* you sell—queen‑size sheet sets—and position your company as a premium, comfortable, and trustworthy option.  
Pick the one that feels most “you,” or mix and match elements that resonate.

| # | Name | Why it works | Quick tagline |
|---|------|--------------|---------------|
| 1 | **QueenComfort Sheets** | “Queen” signals size & royalty; “Comfort” gives the core benefit | “Sleep Like Royalty” |
| 2 | **Royal Rest Bedding** | Royal implies luxury; Rest signals relaxation | “Royalty Redefined” |
| 3 | **Velvet Crown Sheets** | Velvet evokes softness; Crown signals superior quality | “Feel the Velvet of a Crown” |
| 4 | **Elite Queen Bedding** | Elite conveys exclusivity; Queen clarifies size | “Premium Sheets for the Queen in You” |
| 5 | **CrownSoft Queen Sets** | Softness + Crown – instantly suggests luxurious touch | “Crown Your Sleep” |
| 6 | **SuiteQueen Sheets** | “Suite” hints at a full bedding set; Queen as

## `SimpleSequentialChain` equivalent

Single input, single output, piped straight from one chain into the next.

In [6]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain_one = first_prompt | model | StrOutputParser()

second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 word description for the following company: {company_name}"
)
chain_two = second_prompt | model | StrOutputParser()

overall_simple_chain = chain_one | (lambda company_name: {"company_name": company_name}) | chain_two

print(overall_simple_chain.invoke({"product": product}, config=traced("L3: SimpleSequentialChain equivalent")))

A boutique sheet‑set brand blending luxurious, size‑specific bedding with royal elegance, delivering comfort for discerning queens worldwide nightly every season.


## `SequentialChain` equivalent

Multiple named inputs/outputs: translate a review, summarize it, detect its language, then write
a follow-up in that language - each step reads keys earlier steps added and adds its own.
`RunnablePassthrough.assign(key=chain)` is the LCEL building block: it runs `chain` on the
current dict and merges its output back in under `key`, so the dict keeps growing.

In [14]:
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{review}"
)
chain_translate = translate_prompt | model | StrOutputParser()

summarize_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_summarize = summarize_prompt | model | StrOutputParser()

language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{review}"
)
chain_language = language_prompt | model | StrOutputParser()

followup_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_followup = followup_prompt | model | StrOutputParser()

overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_translate)
    | RunnablePassthrough.assign(summary=chain_summarize)
    | RunnablePassthrough.assign(language=chain_language)
    | RunnablePassthrough.assign(followup_message=chain_followup)
)

In [15]:
review = df.Review[5]
result = overall_chain.invoke({"review": review}, config=traced("L3: SequentialChain equivalent"))
for key in ("English_Review", "summary", "language", "followup_message"):
    print(f"\n{key}:\n{result[key]}")


English_Review:
I find the taste mediocre. The mousse doesn’t hold—it's weird. I buy the same thing in the store, and the taste is much better… Old batch or counter‑feiting!?

summary:
The product offers a mediocre taste, a mousse that doesn’t hold, and overall lower quality than the store version, raising doubts it might be an old or counterfeit batch.

language:
That review is in **French**.

followup_message:
Cher(e) client(e),

Nous vous remercions d’avoir pris le temps de partager votre expérience. Nous sommes sincèrement désolés d’apprendre que le produit que vous avez reçu ne répond pas à vos attentes – tant en termes de goût que de texture, et que vous avez constaté une différence notable par rapport à la version en magasin.

Cette situation ne correspond pas aux standards de qualité que nous nous efforçons de maintenir. Il est possible que la boîte ait été affectée par un lot vieillissant ou qu’elle ne provienne pas du dernier produit authentifié. Nous prenons votre commentai

## Router chain equivalent

Four specialist prompts (physics, math, history, computer science) plus a default. The original
notebook has the LLM emit a markdown-wrapped JSON blob naming the destination, hand-parsed by
`RouterOutputParser`. The modern version just asks for a `RouterDecision` via
`with_structured_output` - the model directly returns a validated `destination` +
`next_input`, no string parsing involved.

In [16]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise \
and easy to understand manner. When you don't know the answer to a \
question you admit that you don't know.

Here is a question:
{input}"""

math_template = """You are a very good mathematician. \
You are great at answering math questions. You are so good because \
you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the \
broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people, events \
and contexts from a range of historical periods. You have the ability \
to think, reflect, debate, discuss and evaluate the past. You have a \
respect for historical evidence and the ability to make use of it to \
support your explanations and judgements.

Here is a question:
{input}"""

computerscience_template = """You are a successful computer scientist. \
You have a passion for creativity, collaboration, forward-thinking, \
confidence, strong problem-solving capabilities, understanding of \
theories and algorithms, and excellent communication skills. You are \
great at answering coding questions.

Here is a question:
{input}"""

prompt_infos = [
    {"name": "physics", "description": "Good for answering questions about physics", "template": physics_template},
    {"name": "math", "description": "Good for answering math questions", "template": math_template},
    {"name": "History", "description": "Good for answering history questions", "template": history_template},
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "template": computerscience_template,
    },
]

In [18]:
destination_chains = {
    info["name"]: ChatPromptTemplate.from_template(info["template"]) | model | StrOutputParser()
    for info in prompt_infos
}
default_chain = ChatPromptTemplate.from_template("{input}") | model | StrOutputParser()

destinations_str = "\n".join(f"{info['name']}: {info['description']}" for info in prompt_infos)


class RouterDecision(BaseModel):
    """Pick which specialist prompt should answer the question."""

    destination: str = Field(
        description=f"One of the candidate prompt names, or DEFAULT if none fit well:\n{destinations_str}"
    )
    next_input: str = Field(description="The question to send to that destination, possibly reworded for clarity")


router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given a raw text input to a language model, select the model prompt best suited "
            "for the input. You may also revise the original input if you think that revising it "
            "will ultimately lead to a better response from the language model.\n\n"
            f"Candidate prompts:\n{destinations_str}",
        ),
        ("user", "{input}"),
    ]
)
router_chain = router_prompt | model.with_structured_output(RouterDecision, method="function_calling")


def route(decision: RouterDecision, config=None) -> str:
    chain = destination_chains.get(decision.destination, default_chain)
    return chain.invoke({"input": decision.next_input}, config=config)

In [19]:
for question in ["What is black body radiation?", "what is 2 + 2", "Why does every cell in our body contain DNA?"]:
    decision = router_chain.invoke({"input": question}, config=traced(f"L3: router decision - {question[:30]}"))
    print(f"\n> {question}")
    print(f"routed to: {decision.destination!r}")
    print(route(decision, config=traced(f"L3: router answer - {question[:30]}")))


> What is black body radiation?
routed to: 'physics'
**Black‑body radiation** is the electromagnetic radiation that a perfectly absorbing and perfectly emitting solid (or surface) would emit when it is in thermal equilibrium at a given temperature.

| Key idea | What it means |
|----------|---------------|
| **Perfect absorber** | A black body absorbs *all* incident radiation, regardless of wavelength. |
| **Perfect emitter** | In equilibrium, it emits radiation with the *maximum* possible intensity at each wavelength. |
| **Thermal equilibrium** | The body’s temperature is uniform and does not change with time. |

Because of these two properties, the spectrum of a black‑body depends **only** on its temperature, not on its composition or shape.

### Theoretical description

1. **Planck’s law (1900)** – gives the spectral radiance \(B_\lambda(T)\) (energy per unit area, per unit time, per unit solid angle, per unit wavelength):
   \[
   B_\lambda(T) = \frac{2hc^2}{\lambda^5}\frac{1}{e^